In [1]:
pip install hdbscan

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import hdbscan
import time


In [2]:
df_all = pd.read_parquet("phase3_table.parquet")
df_all.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,Level2,Level3,Level4,raw_text,cleaned_text,dataset,embedding,yake_keywords,keybert_keywords,combined_keywords
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...,train,"[-0.0038827166000000002, -0.0356897637, -0.054...","[asus, vivopc, window, desktop, time, faster, ...","[hardware, it049t, menu, save, online, iterati...","[asus, vivopc, window, desktop, time, faster, ..."
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...,train,"[-0.0597511269, -0.051531221700000004, 0.06836...","[keyboard, notebook, spare, pavilion, part, be...","[hp, keyboard, backlight, a41, compatibility, ...","[keyboard, notebook, spare, pavilion, part, be..."
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...,train,"[0.0158797428, -0.0839086846, 0.03860475500000...","[cable, fiber, patch, connector, duplex, singl...","[fibre, gigabit, duplex, plenum, multicolored,...","[cable, fiber, patch, connector, duplex, singl..."
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...,train,"[-0.08276291940000001, 0.0492046289, 0.0070898...","[battery, mah, lithium-ion, li-ion, product, w...","[hp, fa889aa, lithium, type, spare, power, ipa...","[battery, mah, lithium-ion, li-ion, product, w..."
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...,train,"[-0.0086110868, -0.046555958700000004, -0.0559...","[workstation, professional, thinkstation, memo...","[lenovo, 300gb, e5, thinkstation, dual, certif...","[workstation, professional, thinkstation, memo..."


In [4]:
X_norm = np.load("phase3_embeddings.npy")
X_norm

array([[-0.00388272, -0.03568976, -0.05419029, ..., -0.02733339,
        -0.04840814,  0.01282197],
       [-0.05975113, -0.05153122,  0.0683663 , ...,  0.04304725,
         0.02554441, -0.03286568],
       [ 0.01587974, -0.08390868,  0.03860475, ..., -0.02152328,
         0.01063297, -0.01703629],
       ...,
       [-0.04476147, -0.0154138 ,  0.01855075, ..., -0.03180898,
         0.02299018, -0.07666707],
       [-0.03871163, -0.07086224,  0.06322224, ..., -0.07342245,
         0.06431238, -0.01605557],
       [ 0.01273595, -0.03390823, -0.03061707, ..., -0.01859044,
        -0.0550065 ,  0.02872895]], dtype=float32)

In [5]:
print("Data loaded:", df_all.shape)
print("Embedding matrix shape:", X_norm.shape)

Data loaded: (642997, 20)
Embedding matrix shape: (642997, 384)


In [6]:
# ---------------------------
# 2) Dimensionality Reduction with PCA
# ---------------------------
# WHAT: Reduce from 384 dims → 100 dims
# WHY: Speeds up clustering, removes noise, avoids overfitting to embedding quirks
pca = PCA(n_components=80, random_state=42)

start_pca = time.time()
X_pca = pca.fit_transform(X_norm)
end_pca = time.time()

print(f"PCA took {end_pca - start_pca:.2f} sec")
print("PCA output shape:", X_pca.shape)


PCA took 1.85 sec
PCA output shape: (642997, 80)


In [7]:
# ---------------------------
# 3) HDBSCAN clustering
# ---------------------------
# WHAT: Density-based clustering that handles noise (-1 label)
# WHY: Finds variable-sized clusters without asking for k
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,                  # tweak for granularity
    metric='euclidean',                   # PCA → Euclidean works well
    cluster_selection_epsilon=0.10,        # tolerance for cluster merging )smallest MST edge vcalled gap or bridge)
    cluster_selection_method='eom',        # "excess of mass" (stable clusters)
    prediction_data=True
)

In [8]:
start_hdb = time.time()
labels = clusterer.fit_predict(X_pca)
end_hdb = time.time()

print(f"HDBSCAN took {end_hdb - start_hdb:.2f} sec")
print("Unique clusters found:", len(set(labels)))


HDBSCAN took 36806.77 sec
Unique clusters found: 3383


In [9]:
# 4) Attach cluster labels
# ---------------------------
df_all["HDBSCAN_Cluster"] = labels

In [10]:
# Optional: human-readable label mapping
def cluster_label(n):
    return f"C{n}" if n >= 0 else "Noise"

df_all["Cluster_Label"] = df_all["HDBSCAN_Cluster"].apply(cluster_label)


In [11]:
# 5) Save Phase 4 output
# ---------------------------
df_all.to_json("phase4_pca_clusters.json", orient="records", lines=True)
print("Phase 4 complete — clusters saved")
print(df_all["HDBSCAN_Cluster"].value_counts().head())


Phase 4 complete — clusters saved
HDBSCAN_Cluster
-1       263262
 2706     20274
 2357     11021
 2311     10239
 2730      4714
Name: count, dtype: int64
